In [0]:
%python
import os
import time
import json
import random
from datetime import datetime
from faker import Faker

fake = Faker()
output_dir = "/Volumes/databrickstask/raw_files/source_data/drivers/"
source_path = "/Volumes/databrickstask/raw_files/source_data/drivers/"
checkpoint_path = "/Volumes/databrickstask/raw_files/source_data/checkpoints_files/"
target_table = "databrickstask.bronze.drivers"
os.makedirs(output_dir, exist_ok=True)

def event_stream(delay=0.5):
    while True:
        event = {
            "name": fake.name(),
            "car_number": fake.license_plate(),
            "experience": random.randint(1, 30),
            "rating": round(random.uniform(0, 5.0), 2),
            "phone": fake.phone_number(),
            "timestamp": datetime.now().isoformat()
        }
        yield event
        time.sleep(delay)

for i, event in enumerate(event_stream(delay=1), start=1):
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    filename = f"{output_dir}event_{timestamp_str}.json"
    
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(event, f, ensure_ascii=False)
    if i>=5:
        break


df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json") 
    .option("ignoreMissingFiles", "true")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(source_path)
)

query = (
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option("mergeSchema", "true") 
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)# .trigger(processingTime="10 seconds")
    .toTable(target_table)
)
query.awaitTermination()
